In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nibabel as nib
import json
from pathlib import Path

In [ ]:
TARGET_CATEGORIES = {"2c", "2d"}

with open("C:\\Users\\chamu\\D\\UOR\\FYP\\fyp\\data\\rexgrounding-ct\\dataset.json", "r") as f:
    data = json.load(f)

filtered_data = {}

for split in ["train", "val", "test"]:
    filtered_data[split] = []

    for sample in data.get(split, []):

        categories = sample.get("categories", {})

        keep_keys = [
            k for k, v in categories.items()
            if v in TARGET_CATEGORIES
        ]

        # Skip studies with no 2c/2d findings
        if not keep_keys:
            continue

        new_sample = sample.copy()

        # Filter only fields that exist
        for field in ["findings", "entity_counts", "pixels", "categories"]:
            if field in sample:
                new_sample[field] = {
                    k: sample[field][k]
                    for k in keep_keys
                    if k in sample[field]
                }

        filtered_data[split].append(new_sample)

with open("C:\\Users\\chamu\\D\\UOR\\FYP\\fyp\\data\\rexgrounding-ct\\dataset_2.json", "w") as f:
    json.dump(filtered_data, f, indent=4)

In [ ]:
from collections import defaultdict

stats = defaultdict(int)

for split in ["train", "val", "test"]:
    split_stats = {"2c_only": 0, "2d_only": 0, "both": 0}

    for sample in data.get(split, []):
        categories = set(sample.get("categories", {}).values())

        has_2c = "2c" in categories
        has_2d = "2d" in categories

        if has_2c and has_2d:
            split_stats["both"] += 1
        elif has_2c:
            split_stats["2c_only"] += 1
        elif has_2d:
            split_stats["2d_only"] += 1

    print(f"\n{split.upper()}")
    print(f"  2c only: {split_stats['2c_only']}")
    print(f"  2d only: {split_stats['2d_only']}")
    print(f"  both   : {split_stats['both']}")
    print(f"  total  : {sum(split_stats.values())}")

    for k, v in split_stats.items():
        stats[k] += v

print("\nOVERALL")
print(f"  2c only: {stats['2c_only']}")
print(f"  2d only: {stats['2d_only']}")
print(f"  both   : {stats['both']}")
print(f"  total  : {sum(stats.values())}")

In [ ]:
import json

with open("C:\\Users\\chamu\\D\\UOR\\FYP\\fyp\\data\\rexgrounding-ct\\dataset_2.json", "r") as f:
    data = json.load(f)

print("Train:", len(data["train"]))
print("Val:", len(data["val"]))
print("Test:", len(data["test"]))

print("Total:", len(data["train"]) + len(data["val"]) + len(data["test"]))

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Load the dataset
dataset_path = Path("/home/chest_ct/code/data/rexgrounding-ct/dataset_3.json")
with open(dataset_path, 'r') as f:
    data = json.load(f)

# Filter entries with normal lung volumes (empty categories)
normal_lungs = [item for item in data['train'] if item.get('categories') == {}]

print(f"Total normal lung volumes found: {len(normal_lungs)}")

# Get first 100 normal lung volumes
normal_lungs_100 = normal_lungs[:100]
print(f"Taking first 100 normal lung volumes: {len(normal_lungs_100)}")

# Check how many have shape [32, 32, 32]
size_32x32x32 = [item for item in normal_lungs_100 if item.get('shape') == [32, 32, 32]]
print(f"Count with shape [32, 32, 32]: {len(size_32x32x32)}")

# Show sample of normal lungs
df_normal = pd.DataFrame({
    'name': [item['name'] for item in normal_lungs_100],
    'shape': [item.get('shape', []) for item in normal_lungs_100],
    'findings': [len(item.get('findings', {})) for item in normal_lungs_100],
    'protocol': [item.get('protocol', '') for item in normal_lungs_100]
})

print("\nSample of 100 normal lung volumes:")
print(df_normal.head(10))
print(f"\nShape distribution:")
print(df_normal['shape'].value_counts().head(10))

# Voxel distribution

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Load RexGround-CT dataset.json
# ============================================================

json_path = r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset.json"

with open(json_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)


# ============================================================
# Extract all findings from train/valid/test
# ============================================================

records = []

for split in ["train", "valid", "test"]:

    scans = dataset.get(split, [])

    print(f"{split}: {len(scans)} scans")

    for scan in scans:

        scan_name = scan["name"]

        findings = scan.get("findings", {})
        pixels = scan.get("pixels", {})
        entities = scan.get("entity_counts", {})
        categories = scan.get("categories", {})


        for idx, finding in findings.items():

            records.append({

                "split": split,

                "scan": scan_name,

                "finding": finding,

                "category": categories.get(idx, None),

                "voxels": pixels.get(idx, 0),

                "entities": entities.get(idx, 0)

            })


df = pd.DataFrame(records)


print("\n================================")
print("Dataset Statistics")
print("================================")

print("Total scans:",
      sum(len(dataset.get(s, [])) for s in ["train","valid","test"]))

print("Total findings:",
      len(df))


df.head()

In [ ]:
# ============================================================
# Analyze category 2D and 2C separately
# ============================================================

for target_category in ["2d", "2c"]:

    print("\n" + "="*60)
    print(f"Category: {target_category}")
    print("="*60)


    cat_df = df[df["category"].str.lower() == target_category]


    print(f"Number of annotations : {len(cat_df)}")
    print(f"Total voxels          : {cat_df['voxels'].sum():,}")
    print(f"Mean voxels           : {cat_df['voxels'].mean():,.2f}")
    print(f"Median voxels         : {cat_df['voxels'].median():,.2f}")
    print(f"Minimum voxels        : {cat_df['voxels'].min():,}")
    print(f"Maximum voxels        : {cat_df['voxels'].max():,}")


    print("\nPercentiles:")
    for p in [5, 25, 50, 75, 90, 95, 99]:
        print(
            f"P{p}: {np.percentile(cat_df['voxels'], p):,.0f}"
        )


    # ----------------------------
    # Histogram
    # ----------------------------

    plt.figure(figsize=(9,4))

    plt.hist(
        cat_df["voxels"],
        bins=50
    )

    plt.xlabel("Voxel count")
    plt.ylabel("Number of annotations")

    plt.title(
        f"Voxel Distribution - Category {target_category}"
    )

    plt.yscale("log")

    plt.grid(alpha=0.3)

    plt.show()


    # ----------------------------
    # Boxplot
    # ----------------------------

    plt.figure(figsize=(8,2))

    plt.boxplot(
        cat_df["voxels"],
        vert=False
    )

    plt.xscale("log")

    plt.xlabel("Voxel count (log scale)")

    plt.title(
        f"Voxel Size Distribution - {target_category}"
    )

    plt.grid(alpha=0.3)

    plt.show()

In [ ]:
# ============================================================
# Clean category values
# ============================================================

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# Check available categories
print("Available categories:")
print(df["category"].value_counts())


# ============================================================
# Analyze 2c and 2d separately
# ============================================================

for target_category in ["2c", "2d"]:

    print("\n" + "="*70)
    print(f"ANALYSIS FOR CATEGORY: {target_category.upper()}")
    print("="*70)


    cat_df = df[
        df["category"] == target_category
    ].copy()


    if len(cat_df) == 0:
        print("No annotations found")
        continue


    # --------------------------------------------------------
    # Basic annotation statistics
    # --------------------------------------------------------

    print("\nAnnotation statistics")

    print(
        "Number of annotations:",
        len(cat_df)
    )

    print(
        "Number of unique scans:",
        cat_df["scan"].nunique()
    )

    print(
        "Total entities:",
        cat_df["entities"].sum()
    )

    print(
        "Total voxels:",
        f"{cat_df['voxels'].sum():,}"
    )


    # --------------------------------------------------------
    # Voxel statistics
    # --------------------------------------------------------

    print("\nVoxel statistics")

    print(
        f"Mean voxels:   {cat_df['voxels'].mean():,.2f}"
    )

    print(
        f"Median voxels: {cat_df['voxels'].median():,.2f}"
    )

    print(
        f"Min voxels:    {cat_df['voxels'].min():,}"
    )

    print(
        f"Max voxels:    {cat_df['voxels'].max():,}"
    )


    print("\nPercentiles")

    for p in [1,5,10,25,50,75,90,95,99]:

        print(
            f"P{p}: {np.percentile(cat_df['voxels'],p):,.0f}"
        )


    # --------------------------------------------------------
    # Top largest annotations
    # --------------------------------------------------------

    print("\nLargest annotations")

    display(
        cat_df.sort_values(
            "voxels",
            ascending=False
        )
        .head(10)
        [
            [
                "scan",
                "finding",
                "voxels",
                "entities"
            ]
        ]
    )


    # ========================================================
    # Histogram
    # ========================================================

    plt.figure(figsize=(9,5))

    plt.hist(
        cat_df["voxels"],
        bins=50
    )

    plt.xlabel(
        "Voxel count"
    )

    plt.ylabel(
        "Number of annotations"
    )

    plt.title(
        f"{target_category.upper()} voxel distribution"
    )

    plt.yscale("log")

    plt.grid(alpha=0.3)

    plt.show()



    # ========================================================
    # Log voxel histogram
    # Better for medical lesion sizes
    # ========================================================

    plt.figure(figsize=(9,5))


    # Remove zero voxel annotations before log transform
    positive_voxels = cat_df.loc[
        cat_df["voxels"] > 0,
        "voxels"
    ]


    plt.figure(figsize=(9,5))


    plt.hist(
        np.log10(positive_voxels),
        bins=50
    )


    plt.xlabel(
        "log10(voxel count)"
    )

    plt.ylabel(
        "Number of annotations"
    )


    plt.title(
        f"{target_category.upper()} log voxel distribution"
    )


    plt.grid(alpha=0.3)

    plt.show()


    plt.xlabel(
        "log10(voxel count)"
    )

    plt.ylabel(
        "Number of annotations"
    )


    plt.title(
        f"{target_category.upper()} log voxel distribution"
    )


    plt.grid(alpha=0.3)

    plt.show()



    # ========================================================
    # Boxplot
    # ========================================================

    plt.figure(figsize=(8,2))


    plt.boxplot(
        cat_df["voxels"],
        vert=False
    )


    plt.xscale("log")


    plt.xlabel(
        "Voxel count (log scale)"
    )


    plt.title(
        f"{target_category.upper()} lesion size"
    )


    plt.grid(alpha=0.3)

    plt.show()

In [ ]:
import json
import os
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Paths
# ============================================================

json_path = "/home/chest_ct/code/data/rexgrounding-ct/dataset_2_last.json"

gt_dir = "/home/chest_ct/code/data/segmentations/segmentations"



# ============================================================
# Load JSON
# ============================================================

with open(json_path, "r") as f:
    dataset = json.load(f)



# ============================================================
# Function to count GT voxels
# ============================================================

def get_gt_voxel_count(mask_path):

    if not os.path.exists(mask_path):
        return None


    mask = nib.load(mask_path).get_fdata()


    # If finding dimension exists
    if mask.ndim == 4:
        mask = mask[0]


    return np.count_nonzero(mask > 0)



# ============================================================
# Extract only 2c and 2d cases
# ============================================================

records = []


for split in ["train", "test"]:

    for case in dataset[split]:

        category = case.get("categories", {}).get("1")


        # Ignore normal / empty
        if category not in ["2c", "2d"]:
            continue


        name = case["name"]


        mask_path = os.path.join(
            gt_dir,
            name
        )


        voxel_count = get_gt_voxel_count(
            mask_path
        )


        records.append(
            {
                "split": split,
                "name": name,
                "category": category,
                "gt_voxels": voxel_count
            }
        )



# ============================================================
# Dataframe
# ============================================================

df = pd.DataFrame(records)


print(df.head())


print("\nCases:")
print(
    df.groupby(
        ["split","category"]
    ).size()
)



print("\nMissing GT files:")
print(
    df["gt_voxels"].isna().sum()
)


df = df.dropna()



# ============================================================
# Overall statistics
# ============================================================

for split in ["train","test"]:

    for category in ["2c","2d"]:

        values = df[
            (df["split"] == split) &
            (df["category"] == category)
        ]["gt_voxels"]


        if len(values)==0:
            continue


        print("\n================================")
        print(
            f"{split.upper()} - {category}"
        )
        print("================================")


        print(
            "Cases:",
            len(values)
        )

        print(
            "Total GT voxels:",
            int(values.sum())
        )

        print(
            "Mean:",
            values.mean()
        )

        print(
            "Median:",
            values.median()
        )

        print(
            "Std:",
            values.std()
        )

        print(
            "Min:",
            values.min()
        )

        print(
            "Max:",
            values.max()
        )


        print("\nPercentiles:")

        print(
            values.quantile(
                [
                    0.01,
                    0.05,
                    0.25,
                    0.50,
                    0.75,
                    0.95,
                    0.99
                ]
            )
        )



# ============================================================
# Save results
# ============================================================

df.to_csv(
    "gt_voxel_distribution_2c_2d.csv",
    index=False
)


print(
    "\nSaved gt_voxel_distribution_2c_2d.csv"
)



# ============================================================
# Histograms
# ============================================================

for split in ["train","test"]:

    plt.figure(figsize=(10,6))


    for category in ["2c","2d"]:

        values = df[
            (df["split"] == split) &
            (df["category"] == category)
        ]["gt_voxels"]


        if len(values):

            plt.hist(
                values,
                bins=50,
                alpha=0.5,
                label=category
            )


    plt.xlabel(
        "GT positive voxel count"
    )

    plt.ylabel(
        "Number of cases"
    )

    plt.title(
        f"{split} GT voxel distribution (2c vs 2d)"
    )


    # Lesion sizes vary widely
    plt.xscale("log")
    plt.yscale("log")


    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
import json
from collections import Counter

# Dataset path
dataset_path = r"/home/chest_ct/code/data/rexgrounding-ct/dataset_2.json"

# Load dataset
with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

splits = ["train", "val", "test"]

only_2c_cases = []
entity_counter = Counter()

for split in splits:
    for case in dataset.get(split, []):
        categories = case.get("categories", {})

        # Skip scans without annotations
        if len(categories) == 0:
            continue

        unique_categories = set(categories.values())

        # Keep only scans whose abnormalities are exclusively 2c
        if unique_categories == {"2c"}:
            only_2c_cases.append({
                "split": split,
                "name": case["name"],
                "num_findings": len(categories),
                "findings": case["findings"]
            })

            # Count individual abnormalities
            for finding in case["findings"].values():
                entity_counter[finding] += 1

print("=" * 60)
print(f"Cases containing ONLY category 2c: {len(only_2c_cases)}")
print("=" * 60)

print("\nBreakdown of 2c abnormalities:")
for finding, count in entity_counter.most_common():
    print(f"{count:4d} : {finding}")

print("\nFirst 10 matching cases:")
for case in only_2c_cases[:10]:
    print(f"{case['split']:5s}  {case['name']}  ({case['num_findings']} findings)")

In [ ]:
import json
from pathlib import Path

path = Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

matching_cases = []
ground_glass_cases = []

for split_name, split_cases in data.items():
    if not isinstance(split_cases, list):
        continue

    for case in split_cases:
        categories = list(case.get("categories", {}).values())
        if not categories or set(categories) != {"2c"}:
            continue

        findings_text = " ".join(case.get("findings", {}).values()).lower()
        matching_cases.append(case)

        if "ground" in findings_text and "glass" in findings_text:
            ground_glass_cases.append(case)

print("Cases with only 2c as abnormality:", len(matching_cases))
print("Of those, cases containing both 'ground' and 'glass' in findings:", len(ground_glass_cases))

print("\nExamples:")
for case in ground_glass_cases[:20]:
    print(case.get("name"))

In [ ]:
import json
from pathlib import Path

path = Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

matching_cases = []
nodule_cases = []

for split_name, split_cases in data.items():
    if not isinstance(split_cases, list):
        continue

    for case in split_cases:
        categories = list(case.get("categories", {}).values())
        if not categories or set(categories) != {"2c"}:
            continue

        findings_text = " ".join(case.get("findings", {}).values()).lower()
        matching_cases.append(case)

        if "nodule" in findings_text or "nodular" in findings_text:
            nodule_cases.append(case)

print("Cases with only 2c as abnormality:", len(matching_cases))
print("Of those, cases containing 'nodule' or 'nodular' in findings:", len(nodule_cases))

print("\nExamples:")
for case in nodule_cases[:20]:
    print(case.get("name"))

In [ ]:
import json
from pathlib import Path

path = Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

matching_cases = []
nodule_cases = []

for split_name, split_cases in data.items():
    if not isinstance(split_cases, list):
        continue

    for case in split_cases:
        categories = list(case.get("categories", {}).values())
        if not categories or set(categories) != {"2c"}:
            continue

        findings_text = " ".join(case.get("findings", {}).values()).lower()
        matching_cases.append(case)

        if "consolidation" in findings_text:
            nodule_cases.append(case)

print("Cases with only 2c as abnormality:", len(matching_cases))
print("Of those, cases containing 'consolidation' in findings:", len(nodule_cases))

print("\nExamples:")
for case in nodule_cases[:20]:
    print(case.get("name"))

In [ ]:
import json
from pathlib import Path

path = Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)

matching_cases = []
nodule_cases = []

for split_name, split_cases in data.items():
    if not isinstance(split_cases, list):
        continue

    for case in split_cases:
        categories = list(case.get("categories", {}).values())
        if not categories or set(categories) != {"2c"}:
            continue

        findings_text = " ".join(case.get("findings", {}).values()).lower()
        matching_cases.append(case)

        if "mosaic" in findings_text:
            nodule_cases.append(case)

print("Cases with only 2c as abnormality:", len(matching_cases))
print("Of those, cases containing 'mosaic' in findings:", len(nodule_cases))

print("\nExamples:")
for case in nodule_cases[:20]:
    print(case.get("name"))

In [ ]:
import json
from pathlib import Path

path = Path("/home/chest_ct/code/data/rexgrounding-ct/dataset_2.json")

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)


remove_cases = []
keep_cases = []


for split_name, split_cases in data.items():

    if not isinstance(split_cases, list):
        continue

    for case in split_cases:

        categories = list(case.get("categories", {}).values())

        # only 2c cases
        if not categories or set(categories) != {"2c"}:
            continue


        findings_text = " ".join(
            case.get("findings", {}).values()
        ).lower()


        has_ggo = (
            "ground glass" in findings_text 
            or "ground-glass" in findings_text
        )

        has_nodule = (
            "nodule" in findings_text
            or "nodular" in findings_text
        )

        has_consolidation = (
            "consolidation" in findings_text
        )

        has_mosaic = (
            "mosaic" in findings_text
        )


        # Keep only clean GGO cases
        if (
            has_ggo
            and not has_nodule
            and not has_consolidation
            and not has_mosaic
        ):
            keep_cases.append(case["name"])

        else:
            remove_cases.append(case["name"])


print("Total 2c-only cases:", len(remove_cases) + len(keep_cases))
print("Kept pure GGO cases:", len(keep_cases))
print("Removed cases:", len(remove_cases))


print("\nRemoved examples:")
for c in remove_cases[:20]:
    print(c)

# EDA for all JSONs

In [ ]:
import json
from pathlib import Path
from collections import Counter

dataset_paths = [
    Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2.json"),
    Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2_cleaned.json"),
    Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2_last.json"),
    Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2_ultimate.json"),
    Path(r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2_filtered.json")
]


def count_categories(dataset_path):
    with dataset_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    split_counters = {}

    for split_name, split_cases in data.items():

        if not isinstance(split_cases, list):
            continue

        counter = Counter()

        for case in split_cases:
            categories = list(case.get("categories", {}).values())

            if not categories:
                counter["empty"] += 1
            else:
                key = ", ".join(sorted(set(categories)))
                counter[key] += 1

        split_counters[split_name] = counter

    return split_counters


for dataset_path in dataset_paths:

    split_counters = count_categories(dataset_path)

    print("\n" + "=" * 60)
    print(dataset_path.name)
    print("=" * 60)

    for split_name, counter in split_counters.items():

        print(f"\n--- {split_name} ---")
        print("CT cases by category combination\n")

        for category, count in sorted(counter.items()):
            print(f"{category:40s} : {count}")


        print(f"Total {split_name}: {sum(counter.values())}")

In [ ]:
import json
from pathlib import Path
from collections import Counter

dataset_path = Path(
    r"D:\My\Projects\fyp-3d-ct\data\rexgrounding-ct\dataset_2_last.json"
)

with dataset_path.open("r", encoding="utf-8") as f:
    data = json.load(f)


def find_duplicates(names):
    counter = Counter(names)
    return {k: v for k, v in counter.items() if v > 1}


train_names = [case["name"] for case in data["train"]]
test_names = [case["name"] for case in data["test"]]

train_set = set(train_names)
test_set = set(test_names)

train_dups = find_duplicates(train_names)
test_dups = find_duplicates(test_names)

cross_dups = sorted(train_set & test_set)

all_names = train_names + test_names
all_dups = find_duplicates(all_names)

train_normal = sum(
    case.get("categories", {}) == {}
    for case in data["train"]
)

test_normal = sum(
    case.get("categories", {}) == {}
    for case in data["test"]
)

print("=" * 60)
print("Dataset validation")
print("=" * 60)

print(f"Train cases          : {len(train_names)}")
print(f"Test cases           : {len(test_names)}")
print(f"Total cases          : {len(all_names)}")
print(f"Unique case names    : {len(set(all_names))}")

print("\nNormal cases")
print(f"  Train : {train_normal}")
print(f"  Test  : {test_normal}")

print("\nDuplicate names within train :", len(train_dups))
if train_dups:
    print(list(train_dups.items())[:20])

print("\nDuplicate names within test :", len(test_dups))
if test_dups:
    print(list(test_dups.items())[:20])

print("\nCases present in BOTH train and test :", len(cross_dups))
if cross_dups:
    print(cross_dups[:20])

print("\nDuplicate names anywhere in dataset :", len(all_dups))
if all_dups:
    print(list(all_dups.items())[:20])